#### Silver layer setup
Silver schema setup and imports required libraries and creates `silver_run_id` for the Run

In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime
import uuid

Unique run id for silver pipeline
Create a unique run id for silver layer ingestion


In [0]:
silver_run_id = str(uuid.uuid4())
print(f"current silver run id: {silver_run_id}")

In [0]:
spark.sql("""
          use catalog novacart_catalog
          """)


spark.sql("""
         CREATE SCHEMA IF NOT EXISTS novacart_catalog.silver
          """)

#### Create Silver control table
This tables stores the latest silver layer procesing state for each entity

this helps us to track:
- The latest bronze run already processed by Silver
- Bronze Ingestion timestamp already processed
- How many rows merged into silver tables

In [0]:
spark.sql("""
          CREATE TABLE IF NOT EXISTS novacart_catalog.audit.processing_control (
              layer STRING,
              table_name STRING,
              last_processed_bronze_run_id STRING,
              last_processed_bronze_ingested_at TIMESTAMP,
              rows_merged BIGINT,
              run_status STRING,
              silver_run_id STRING,
              updated_at TIMESTAMP
              )
              USING DELTA
          """)

#### Building Helper Functions which is reusable for silver layer procesing
- `upsert_to_silver()` merges cleaned/trasformed rows into the silver tables
- `get_last_processed_bronze_ingested_at()` reads the silver watermark
- `upsert_silver_control()` updates the silver control table
- `get_incremental_bronze()` reads new bronze rows that silver has not processed yet
- import these functions from silver_job_control.py 

Now lets read the bronze raw data into dataframe, which later used to clean,standadize and transform for silver tables 

In [0]:
# orders_df_raw = spark.sql("SELECT * FROM novacart_catalog.bronze.orders_brz")
# display(orders_df_raw)


In [0]:
# Import the Helper Functions

from src.utils.silver_job_control import (
    upsert_to_silver,get_last_processed_bronze_ingested_at,upsert_silver_control,get_incremental_bronze)

#### oredrs silver processing

In [0]:
# Here we are processing orders bronze rows to silver layer tables

# target_table_location = "s3://novacart/silver/"
# silver_control_table = "novacart_catalog.audit.processing_control"
# orders_slvr = "novacart_catalog.silver.orders_slvr"
# orders_slvr = "orders_slvr"

join_key = "order_id"


orders_inc,last_orders_ingested_at = get_incremental_bronze(spark,"novacart_catalog.silver.orders_slvr","novacart_catalog.bronze.orders_brz")


# Now, count the incremental / New rows which are going to process
orders_inc_count = orders_inc.count()
print(f"Total new rows to process: {orders_inc_count}")


# Now Cleaning and validating the new or incremental rows only
if orders_inc_count > 0:

    # This is to handle the case where the same order is updated multiple times
    # We will keep the latest order record for each order_id
    orders_window = Window.partitionBy(join_key).orderBy(
                                                    F.col("updated_at").cast("timestamp").desc(),
                                                    F.col("bronze_ingested_at").desc()
                                                    )
    
    # Here We start silver layer cleaning and validation of the new or incremental rows only
    orders_df_cleaned = (
        orders_inc
        
        # standadize the order_status value to UPPERCASE to become consistent with the silver layer
        .withColumn("order_status", F.upper(F.trim(F.col("order_status"))))
        .withColumn(
            "order_status",
            F.when(F.col("order_status").isNull() | (F.trim(F.col("order_status")) == ""),F.lit(None))
            .otherwise(F.col("order_status"))
            )
       
        # Remove Formatting Characters from order_amount, so that we can cast it to Numeric type
        .withColumn("order_amount", F.regexp_replace(F.col("order_amount"), r"[$, ]", ""))
        .withColumn("order_amount", F.when(F.trim(F.col("order_amount")).isin("N/A","??","NULL",""),None).otherwise(F.col("order_amount")))
        .withColumn("order_amount", F.col("order_amount").cast("double"))
        
        # Casting the timestamp columns to timestamp type
        .withColumn("created_at",F.to_timestamp(F.col("created_at")))
        .withColumn("updated_at",F.to_timestamp(F.col("updated_at")))
        .withColumn("bronze_ingested_at",F.to_timestamp(F.col("bronze_ingested_at")))

        # Assign the row_number for ach bussiness key so that we can keep latest version of each bussiness key
        .withColumn("row_rank", F.row_number().over(orders_window))
                    .filter(F.col("row_rank") == 1)
                    .drop("row_rank")
        
        .withColumn("silver_run_id",F.lit(silver_run_id))
    )

    # Merge the cleaned or Validated Silver dataset into its delta table
    upsert_to_silver(spark,orders_df_cleaned,"novacart_catalog.silver.orders_slvr",join_key)

    # Apply silver data quality rules to cleaned orders records
    orders_validated =(
        orders_df_cleaned
        
        # Check for null values in customer_id, product_id, order_status, order_amount columns
        # If any of the columns has null value, then flag it for further investigation
        .withColumn(
            "to_be_verifyed_by_source_team",
            F.when(F.col("customer_id").isNull(),"verify_customer_id")
            .when(F.col("product_id").isNull(),"verify_product_id")
            .when(F.col("order_status").isNull() | (F.trim(F.col("order_status")) == ""),"verify_order_status")
            .when(F.col("order_amount").isNull() | (F.col("order_amount") <= 0),"verify_order_amount")
            .otherwise("No Issues")
        )
        .withColumn(
            "check_order_amount",
            F.when(F.col("order_amount").isNull() | (F.col("order_amount") <= 0),F.lit(True))
            .otherwise(F.lit(False))
        )

        .withColumn("order_date",F.to_date("created_at"))
        .withColumn("order_month",F.month("created_at"))
        .withColumn("order_year",F.year("created_at"))
        .withColumn("order_day",F.dayofmonth("created_at"))
        .withColumn("order_dow",F.date_format("created_at","E"))
     )
    
    # Keep only valid orders rows for transformed silver table
    orders_good = orders_validated.filter(F.col("to_be_verifyed_by_source_team") == "No Issues")
    
    # send invalid orders rows to Qurantine dataset for further investigation
    orders_bad = (
        orders_validated.filter(F.col("to_be_verifyed_by_source_team") != "No Issues")
        .withColumn("quarantine_ts",F.current_timestamp())
    )

    # Merge the cleaned or Validated Silver dataset into its delta table
    upsert_to_silver(spark,orders_good,"novacart_catalog.silver.orders_transformed",join_key)

    # Append bad orders rows to the Quarantine delta table instead of loosing them
    orders_bad.write.format("delta").mode("append").saveAsTable("novacart_catalog.silver.orders_quarantine")

    max_ingested = orders_inc.agg(F.max("bronze_ingested_at").alias("mx")).collect()[0]["mx"]
    
    max_run = (
        orders_inc.filter(F.col("bronze_ingested_at") == F.lit(max_ingested))
                  .agg(F.max("bronze_run_id").alias("mx"))
                  .collect()[0]["mx"]
    )

    upsert_silver_control(
        spark,"novacart_catalog.silver.orders_slvr",max_run,max_ingested,orders_good.count(),silver_run_id)
    
else:
    print("No new rows to process from orders bronze layer")


#### Products Silver Processing

This cell process products from Bronze to Silver
- product Name cleanup
- Catageory Standardization
- Price cleaanup and Numeric Conversion
- latest record selection per `product_id`
- Data Quality Validation
- quarantine for bad rows
- merge into current silver state table

In [0]:
# Lets get the incremental data from bronze layer
# This will return the new rows from bronze layer and the last ingested timestamp

products_brz = "novacart_catalog.bronze.products_brz"

products_slvr = "novacart_catalog.silver.products_slvr"
products_transformed = "novacart_catalog.silver.products_transformed"
products_quarantine = "novacart_catalog.silver.products_quarantine"

join_key = "product_id" 

# Now get the last succesfully ingested timestamp from silver table
# Load the data from last timestamp to get the new rows from bronze layer
products_inc,last_products_ingested_at = get_incremental_bronze(spark,products_slvr,products_brz)


# Now, count the incremental / New rows which we are going to process
products_inc_count = products_inc.count()
print(f"Total new rows to process: {products_inc_count}")

if products_inc_count > 0:

    # Create a window that keeps latest product record for each product_id
    products_window = Window.partitionBy(join_key).orderBy(
                                                    F.col("updated_at").cast("timestamp").desc(),
                                                    F.col("bronze_ingested_at").desc()
                                                    )
    

    # start the silver products transformation of cleaning and validation of the new or incremental rows only
    products_slvr_df = (
        products_inc
        .withColumn("product_name",F.upper(F.trim(F.col("product_name"))))
        .withColumn("product_name",F.regexp_replace(
                                        F.col("product_name"),
                                        r"^PROD\s",
                                        "PRODUCT ")
                    )
        .withColumn("product_name",F.regexp_replace(
                                        F.col("product_name"),
                                        r"[\_\-]",
                                        " ")
                    )
        .withColumn(
            "product_name",
            F.when(F.col("product_name") == " ",F.lit(None))
            .otherwise(F.col("product_name"))
            )
        .withColumn("category",
                    F.when(F.upper(F.trim(F.col("category"))).contains("ELECTRNICS"),"ELECTRONICS")
                    .otherwise(F.upper(F.trim(F.col("category"))))
                    )
        .withColumn("price",F.trim(F.col("price"))) 
        .withColumn("price",F.regexp_replace(F.col("price"),r"\$","")) 
        .withColumn("price",F.regexp_replace(F.col("price"),",",".")) 
        .withColumn("price",F.regexp_replace(F.col("price"),r"\s+","")) 
        .withColumn("price",F.expr("try_cast(price AS DOUBLE)")) 

        #Assign a row number for each bussiness key so that we cn keep only latest verion of that record
        .withColumn("row_rank",F.row_number().over(products_window))
        .filter(F.col("row_rank") == 1)
        .drop("row_number")
        .withColumn("silver_run_id",F.lit(silver_run_id))        
        )
    
    # Now Merge the cleand & Standardized rows into sliver delta table 
    upsert_to_silver(spark,products_slvr_df,products_slvr,join_key)

    # Now apply Data quality Checks to the cleaned and standardized data
    products_validated = (
        products_slvr_df
        .withColumn("to_be_verifyed_by_source_team",
                    F.when(F.col("product_name").isNull(),F.lit("verify_product_name"))
                     .when(F.col("category").isNull(),F.lit("verify_category"))
                     .when(F.col("price").isNull() | (F.col("price") <= 0),F.lit("verify_price"))
                     .otherwise("No Issues")
                    )
        .withColumn("check_product_price",
                    F.when(F.col("price").isNull() | (F.col("price") <= 0),F.lit("invalid_price"))
                    .otherwise("valid_price")  
                    )
        )
    
    # Now Apply filter for good and bad records

    products_good = products_validated.filter(
        (F.col("to_be_verifyed_by_source_team") == "No Issues") &
        (F.col("check_product_price") == "valid_price")
        )
    
    if "price_raw" in products_good.columns:
        products_good = products_good.drop("price_raw")
    

    products_bad = products_validated.filter(
                (F.col("to_be_verifyed_by_source_team") != "No Issues") |
                (F.col("check_product_price") == "invalid_price")
            ).withColumn("quarantine_ts",F.current_timestamp())

    # Now write the good records to respective delta table
    upsert_to_silver(spark,products_good,products_transformed,join_key)
    
    # Append the bad records to quarantine table
    products_bad.write.format("delta").mode("append").saveAsTable(products_quarantine)

    max_ingested = products_inc.agg(F.max("bronze_ingested_at").alias("mx")).collect()[0]["mx"]

    max_run = products_inc.filter(F.col("bronze_ingested_at") == max_ingested).agg(F.max("bronze_run_id").alias("mx")).collect()[0]["mx"]

    upsert_silver_control(
        spark,products_slvr,max_run,max_ingested,products_good.count(),silver_run_id)


else:
    print("No new rows to process from products bronze layer")

    # upsert_silver_control(
    #     spark,products_slvr,None,last_products_ingested_at,products_inc_count,silver_run_id)
    



In [0]:
%sql


select * from novacart_catalog.silver.products_transformed;


#### Payments Silver Processing

This Cell process payments from bronze to silver
- it validates `payment_status`,`paid_amount`,`processed_at`
- Then it Validates records, quarantines ba records and merges good records

In [0]:
# Lets get the incremental data from bronze layer
# This will return the new rows from bronze layer and the last ingested timestamp

payments_brz = "novacart_catalog.bronze.payments_brz"

payments_slvr = "novacart_catalog.silver.payments_slvr"
payments_transformed = "novacart_catalog.silver.payments_transformed"
payments_quarantine = "novacart_catalog.silver.payments_quarantine"

join_key = "payment_id" 

# Now get the last succesfully ingested timestamp from silver table
# Load the data from last timestamp to get the new rows from bronze layer
payments_inc,last_payments_ingested_at = get_incremental_bronze(spark,payments_slvr,payments_brz)


# Now, count the incremental / New rows which we are going to process
payments_inc_count = payments_inc.count()
print(f"Total new rows to process: {payments_inc_count}")


if payments_inc_count > 0:

    # Create a window that keeps latest product record for each product_id
    payments_window = Window.partitionBy(join_key).orderBy(
                                                    F.col("processed_at").cast("timestamp").desc(),
                                                    F.col("bronze_ingested_at").desc()
                                                    )
    

    # start the silver products transformation of cleaning and validation of the new or incremental rows only
    payments_slvr_df = (
        payments_inc
        # Start Standardizing payment_status column
        .withColumn("payment_status",F.upper(F.trim(F.col("payment_status"))))
        .withColumn(
            "payment_status",
            F.when(F.col("payment_status") == "",F.lit(None))
            .otherwise(F.col("payment_status"))
            )
        # Start Standardizing paid_amount column
        .withColumn("paid_amount",F.trim(F.col("paid_amount")))
        .withColumn("paid_amount",
                    F.regexp_replace(F.col("paid_amount"),r"\$","")
                    )
        .withColumn("paid_amount",
                    F.regexp_replace(F.col("paid_amount"),r"\s+","")
                    )
        .withColumn("paid_amount",
                    F.regexp_replace(F.col("paid_amount"),",",".")
                    )
        .withColumn("paid_amount",F.expr("try_cast(paid_amount as double)"))
        .withColumn("processed_at",F.to_timestamp(F.col("processed_at")))
        # Assign a row rumber ofr each business key so that we can keep only latest one
        .withColumn("row_rank",F.row_number().over(payments_window))
        .filter(F.col("row_rank") == 1)
        .drop("row_rank")
        .withColumn("silver_run_id",F.lit(silver_run_id))
        )
    
    
    # Now Merge the cleand & Standardized rows into sliver delta table 
    upsert_to_silver(spark,payments_slvr_df,payments_slvr,join_key)

    # Now apply Data quality Checks to the cleaned and standardized data
    payments_validated = (
        payments_slvr_df
        .withColumn("to_be_verifyed_by_source_team",
                    F.when(F.col("order_id").isNull(),F.lit("verify_order_id"))
                     .when(F.col("payment_status").isNull(),F.lit("verify_payment_status"))
                     .when(F.col("paid_amount").isNull() | (F.col("paid_amount") <= 0),F.lit("verify_paid_amount"))
                     .otherwise("No Issues")
                    )
        .withColumn("check_paid_amount",
                    F.when(F.col("paid_amount").isNull() | (F.col("paid_amount") <= 0),F.lit(True))
                    .otherwise(F.lit(False))
                    )
        )
    
    # Now Apply filter for good and bad records
    payments_good = payments_validated.filter(F.col("to_be_verifyed_by_source_team") == "No Issues")
    
    # Now Apply filter for good and bad records

    payments_bad = payments_validated.filter(F.col("to_be_verifyed_by_source_team") != "No Issues") \
                                     .withColumn("quarantine_ts",F.current_timestamp())
    
    
    # Now write the good records to respective delta table
    upsert_to_silver(spark,payments_good,payments_transformed,join_key)
    
    # Append the bad records to quarantine table
    payments_bad.write.format("delta").mode("append").saveAsTable(payments_quarantine)


    max_ingested = payments_inc.agg(F.max("bronze_ingested_at").alias("mx")).collect()[0]["mx"]

    max_run = payments_inc.filter(F.col("bronze_ingested_at") == F.lit(max_ingested)).agg(F.max("bronze_run_id").alias("mx")).collect()[0]["mx"]

    upsert_silver_control(
        spark,payments_slvr,max_run,max_ingested,payments_good.count(),silver_run_id)


else:
    print("No new rows to process from products bronze layer")


In [0]:
print(f"Orders transformed count:{spark.sql(f"select count(*) from novacart_catalog.silver.orders_transformed").collect()[0][0]}")
print(f"payments transformed count:{spark.sql(f"select count(*) from novacart_catalog.silver.payments_transformed").collect()[0][0]}")
print(f"Products transformed count:{spark.sql(f"select count(*) from novacart_catalog.silver.products_transformed").collect()[0][0]}")


display(spark.table("novacart_catalog.audit.processing_control").orderBy("table_name"))
